<font size="6" color='grey'> <b>
Generative KI. Verstehen. Anwenden. Gestalten.
</b></font> </br>

---

<font size="5" color='grey'> <b>
M05a - Übung A1: Textklassifizierung mit LLMs
</b></font> </br>

**Lernziel:** Sentiment-Analyse von Produktbewertungen mit Few-Shot LLM-Klassifizierung durchführen und aspektbasierte Sentiments identifizieren.

## Setup & Environment

Umgebung vorbereiten und erforderliche Module laden.

In [ ]:
#@title 🔧 Umgebung einrichten (LOCAL VERSION)
# LOKAL: genai_lib muss bereits installiert sein
# Falls nicht: pip install -e /Users/wagnerg/Development/playground/GenAI_GW/lessons/GenAI/04_modul

import subprocess
import sys

# Erforderliche Packages sicherstellen
required_packages = {
    'dotenv': 'python-dotenv',
    'pandas': 'pandas'
}

for module, package in required_packages.items():
    try:
        __import__(module)
    except ImportError:
        print(f"📦 Installiere {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

from dotenv import load_dotenv
import os

# API Keys aus .env laden
env_path = '/Users/wagnerg/Development/playground/GenAI_GW/.env'
load_dotenv(env_path)

# Imports
from genai_lib.utilities import check_environment, mprint

print("✅ Umgebung wird vorbereitet...")
print()
check_environment()
print()
print(f"✓ OPENAI_API_KEY gesetzt: {'OPENAI_API_KEY' in os.environ and os.environ['OPENAI_API_KEY'] != ''}")
print(f"✓ genai_lib importiert erfolgreich")
print(f"✓ pandas verfügbar")

## Imports

Erforderliche LangChain-Komponenten importieren.

In [ ]:
# Importe für Text-Klassifizierung
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers.string import StrOutputParser
from langchain_core.messages import SystemMessage
from pandas import DataFrame
import json

## Model-Konfiguration

Parameter und Modell-Initialisierung.

In [ ]:
# Parameter
model_provider = "openai"
model_name = "gpt-4o-mini"
temperature = 0.3

# Modell definieren
llm = init_chat_model(model_name, model_provider=model_provider, temperature=temperature)

print(f"✅ Modell initialisiert: {model_name}")
print(f"   Temperature: {temperature}")

# Textklassifizierung mit LLMs

Große Sprachmodelle (LLMs) bieten eine flexible Alternative zur klassischen Textklassifizierung. Mit **Few-Shot Learning** können sie Texte basierend auf wenigen Beispielen in vordefinierte Klassen einordnen.

**Vorteile:**
- Keine aufwendigen Trainingsdatensätze nötig
- Schnelle Anpassung an neue Kategorien
- Hohe Genauigkeit durch Verständnis von Kontext

## Beispiel 1: Basis-Klassifizierung (Spam vs. Ham)

Klassische SMS-Spam-Erkennung mit Few-Shot Prompting.

In [ ]:
# System Prompt für SMS-Klassifizierung
system_prompt_sms = """
Du bist ein Experte für die Klassifikation von SMS-Nachrichten.

Klassifiziere SMS-Nachrichten als:
- spam: Werbung, Gewinnspiele, kostenpflichtige Dienste, unerwünschte Nachrichten
- ham: Normale Kommunikation, persönliche Nachrichten

Wichtig: Antworte NUR mit einem Wort: spam oder ham
"""

# Prompt-Template
prompt_sms = ChatPromptTemplate.from_messages([
    ("system", system_prompt_sms),
    ("human", "SMS: {sms_text}")
])

# Parser und Chain
parser = StrOutputParser()
chain_sms = prompt_sms | llm | parser

# Test-SMS
test_sms_list = [
    "WINNER!! As a valued customer you have won £1000 prize!",
    "Hey, can you call me when you get home?",
    "FREE! Claim your prize now! Click: bit.ly/prize123",
    "I'll be home in 10 minutes",
    "Exclusive offer: Buy now, pay later!"
]

mprint("## 📱 SMS-Klassifizierung (Spam vs. Ham)")
mprint("---")

results_sms = []
for sms in test_sms_list:
    result = chain_sms.invoke({"sms_text": sms})
    results_sms.append({"SMS": sms[:50] + "..." if len(sms) > 50 else sms, "Klasse": result.strip()})
    mprint(f"**SMS:** {sms[:60]}..." if len(sms) > 60 else f"**SMS:** {sms}")
    mprint(f"**Klassifizierung:** `{result.strip()}`")
    print()

# Zusammenfassung als Tabelle
df_sms = DataFrame(results_sms)
print(df_sms.to_string(index=False))

## Beispiel 2: Sentiment-Analyse von Produktbewertungen

Klassifizierung mit Aspekt-Analyse (Quality, Price, Delivery, Service).

In [ ]:
# System Prompt für Sentiment-Analyse
system_prompt_sentiment = """
Du bist ein Experte für Sentiment-Analyse von Produktbewertungen.

Analysiere die Bewertung hinsichtlich:
1. Gesamtsentiment: Positiv / Neutral / Negativ
2. Erwähnte Aspekte: Qualität, Preis, Lieferung, Service
3. Sentiment pro Aspekt: Positiv / Negativ / Nicht erwähnt

Antworte in folgendem Format:
GESAMT: [Sentiment]
QUALITÄT: [Sentiment]
PREIS: [Sentiment]
LIEFERUNG: [Sentiment]
SERVICE: [Sentiment]
"""

# Prompt-Template
prompt_sentiment = ChatPromptTemplate.from_messages([
    ("system", system_prompt_sentiment),
    ("human", "Bewertung: {review}")
])

# Chain
chain_sentiment = prompt_sentiment | llm | parser

# Test-Bewertungen
reviews = [
    "Die Qualität des Produkts ist hervorragend, allerdings finde ich den Preis zu hoch.",
    "Schnelle Lieferung, guter Service, faire Preise - besser geht es nicht!",
    "Nach zwei Wochen ging das Gerät kaputt. Der Kundenservice war bei der Reklamation leider keine Hilfe.",
    "Durchschnittliche Qualität, erfüllt seinen Zweck. Lieferung dauerte etwas länger als angegeben."
]

mprint("## ⭐ Sentiment-Analyse von Produktbewertungen")
mprint("---")

results_sentiment = []
for i, review in enumerate(reviews, 1):
    mprint(f"### Bewertung {i}")
    mprint(f"**Text:** {review}")
    
    result = chain_sentiment.invoke({"review": review})
    
    mprint(f"**Analyse:**")
    mprint(result)
    print()
    
    results_sentiment.append({
        "Bewertung_ID": i,
        "Text": review[:40] + "...",
        "Analyse": result.strip()
    })

## Beispiel 3: Zero-Shot Klassifizierung (ohne Beispiele)

Klassifizierung nur basierend auf System-Anweisung, ohne explizite Trainingsbeispiele.

In [ ]:
# System Prompt für Produktkategorie-Klassifizierung
system_prompt_category = """
Klassifiziere Produktbeschreibungen in eine der folgenden Kategorien:
- Elektronik
- Kleidung
- Lebensmittel
- Möbel
- Bücher

Antworte nur mit der Kategorie, keine Erklärung.
"""

prompt_category = ChatPromptTemplate.from_messages([
    ("system", system_prompt_category),
    ("human", "Produkt: {product}")
])

chain_category = prompt_category | llm | parser

# Test-Produkte
products = [
    "Kabelloses Kopfhörer-Set mit Noise-Cancelling und 30 Stunden Akkulaufzeit",
    "Bio-Baumwoll-T-Shirt in verschiedenen Farben, 100% nachhaltig produziert",
    "Premium Arabica Kaffeebohnen, frisch geröstet, 1kg Packung",
    "Ergonomischer Schreibtischstuhl mit verstellbarer Sitzhöhe und Armlehnen",
    "Klassiker der Literatur: Moby Dick Neuausgabe mit Illustrationen"
]

mprint("## 🏷️ Zero-Shot Produktkategorisierung")
mprint("---")

results_category = []
for product in products:
    category = chain_category.invoke({"product": product})
    results_category.append({"Produkt": product[:40] + "...", "Kategorie": category.strip()})
    mprint(f"**Produkt:** {product[:50]}..." if len(product) > 50 else f"**Produkt:** {product}")
    mprint(f"**Kategorie:** `{category.strip()}`")
    print()

# Zusammenfassung als Tabelle
df_category = DataFrame(results_category)
print(df_category.to_string(index=False))

## Beispiel 4: Multi-Klassen-Klassifizierung mit Konfidenz

Erweiterte Klassifizierung mit Begründung und Konfidenz-Score.

In [ ]:
# System Prompt mit ausführlicheren Ausgaben
system_prompt_detailed = """
Analysiere den Text und gib folgende Informationen zurück:
1. Sentiment: Positiv / Neutral / Negativ
2. Konfidenz: Hoch / Mittel / Niedrig
3. Hauptthema: Ein Wort beschreibt das Hauptthema
4. Begründung: 1-2 Sätze Erklärung

Format:
SENTIMENT: [Wert]
KONFIDENZ: [Wert]
THEMA: [Wert]
BEGRÜNDUNG: [Wert]
"""

prompt_detailed = ChatPromptTemplate.from_messages([
    ("system", system_prompt_detailed),
    ("human", "Text: {text}")
])

chain_detailed = prompt_detailed | llm | parser

# Test-Texte
test_texts = [
    "Ich bin begeistert von meinem neuen Laptop! Die Performance ist unglaublich schnell.",
    "Das Produkt funktioniert, aber es gibt bessere Alternativen am Markt.",
    "Das ist ein Desaster! Der Kundenservice hat sich weigert, mich zu helfen!"
]

mprint("## 🔍 Detaillierte Multi-Klassen-Analyse")
mprint("---")

for i, text in enumerate(test_texts, 1):
    mprint(f"### Analyse {i}")
    mprint(f"**Text:** {text}")
    
    analysis = chain_detailed.invoke({"text": text})
    
    mprint(f"**Ergebnis:**")
    mprint(analysis)
    print()

## Batch-Verarbeitung mehrerer Texte

Effiziente parallele Verarbeitung mehrerer Klassifizierungen.

In [ ]:
# Batch-Inputs für SMS-Klassifizierung
batch_sms_inputs = [
    {"sms_text": "Guten Morgen! Wie geht es dir heute?"},
    {"sms_text": "URGENT: Klicke hier um dein Konto zu bestätigen!"},
    {"sms_text": "Treffen wir uns um 15 Uhr im Café?"},
    {"sms_text": "50% RABATT NUR HEUTE! Jetzt kaufen!"},
    {"sms_text": "Danke für die Hilfe gestern! Das war sehr nett."}
]

mprint("## 📦 Batch-Verarbeitung: SMS-Klassifizierung")
mprint("---")

# batch() für parallele Verarbeitung
batch_results = chain_sms.batch(batch_sms_inputs)

batch_df = DataFrame([
    {
        "SMS": inp["sms_text"][:40] + "...",
        "Klassifizierung": result.strip()
    }
    for inp, result in zip(batch_sms_inputs, batch_results)
])

mprint(batch_df.to_markdown(index=False))

## 💡 Erkenntnisse

### Was macht LLM-Klassifizierung so effektiv?

**1. Kontextverständnis**
- LLMs verstehen nicht nur Wörter, sondern auch Bedeutung und Kontext
- Sarkasmus, Mehrdeutigkeit und kulturelle Nuancen werden erkannt

**2. Few-Shot Learning**
- Mit wenigen Beispielen können neue Kategorien gelernt werden
- Keine aufwendigen Trainingsdatensätze erforderlich

**3. Flexibilität**
- Klassifizierungsschema kann leicht angepasst werden
- Neue Kategorien können spontan hinzugefügt werden

**4. Mehrschicht-Analyse**
- Simultan können mehrere Klassifizierungen durchgeführt werden
- z.B. Gesamtsentiment + Aspekt-basierte Analyse

### Wichtige Parameter

| Parameter | Effekt |
|-----------|--------|
| **Temperature** | Kontrolliert Kreativität vs. Determinismus |
| **max_tokens** | Begrenzt Antwortlänge (für schnellere Responses) |
| **top_p** | Nucleus Sampling für kontrollierte Varianz |

### Best Practices

✅ **Do:**
- Klare, präzise Anweisungen geben
- Format explizit vorgeben (z.B. "Antworte nur mit...")
- Beispiele für komplexe Fälle bereitstellen
- Batch-Verarbeitung für mehrere Texte nutzen

❌ **Don't:**
- Mehrdeutige Kategorien verwenden
- Zu lange oder komplexe Anweisungen geben
- Ohne Temperature-Anpassung kritische Klassifizierungen durchführen
- Großmengen einzeln verarbeiten statt Batch-Processing

## 📝 Zusammenfassung

✅ **Was wir gelernt haben:**

1. **Few-Shot Klassifizierung**: Mit LLMs können Texte mit wenigen Beispielen klassifiziert werden
2. **Sentiment-Analyse**: Gefühle und Meinungen in Texten erkennen und bewerten
3. **Zero-Shot Kategorisierung**: Auch ohne Beispiele können neue Kategorien gelernt werden
4. **Aspekt-basierte Analyse**: Mehrere Aspekte eines Textes simultane analysieren
5. **Batch-Processing**: Effiziente parallele Verarbeitung mehrerer Texte

🚀 **Nächste Schritte:**
- Eigene Klassifizierungskategorien definieren
- Die Prompt-Formulierung optimieren für höhere Genauigkeit
- Hybrid-Ansätze kombinieren (LLM + traditionelle Methoden)
- Klassifizierungsergebnisse evaluieren und verifizieren